# Detecting Spatial Scale with Moran's I Correlogram
### *Crash Example — Introducción al Análisis de Datos y Programación para el Manejo y Conservación de Recursos Naturales*

---

This notebook demonstrates how to:

1. Generate a **synthetic landscape** with a *known* spatial autocorrelation scale (σ)
2. **Sample** random points from that landscape
3. **Calculate Moran's I** at multiple lag distances (spatial correlogram)
4. **Visualise** the correlogram
5. **Recover** the characteristic scale from the data — as if σ were unknown

> **Key insight:** The zero-crossing of the correlogram reveals the spatial scale embedded in the data. This is the core logic behind scale detection in landscape ecology.

## 0 · Install dependencies

Only needed the first time in a fresh Colab runtime.

In [ ]:
!pip install numpy scipy matplotlib libpysal esda --quiet

## 1 · Imports

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt

## 2 · Generate a spatially autocorrelated landscape

We start from **white noise** (no spatial structure) and apply a **Gaussian filter** with standard deviation σ (`SIGMA`).  
This smoothing *injects* spatial autocorrelation at a known characteristic scale — the value we will later try to recover.

| Parameter | Meaning |
|-----------|---------|
| `n` | Grid size (n × n cells) |
| `SIGMA` | Characteristic scale in grid cells — the "truth" |

In [ ]:
np.random.seed(42)
n = 100        # 100 × 100 grid

# White noise — no spatial structure
white_noise = np.random.randn(n, n)

# Gaussian smoothing creates spatial autocorrelation at scale SIGMA
SIGMA = 15     # grid cells  ← this is what we want to RECOVER later
landscape = gaussian_filter(white_noise, sigma=SIGMA)

print(f"Landscape shape: {landscape.shape}")
print(f"True characteristic scale σ = {SIGMA} grid cells")

## 3 · Visualise: white noise vs autocorrelated landscape

Notice how the smoothed landscape has broad, coherent patches — that's spatial autocorrelation.  
The patch size is directly controlled by σ.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(white_noise, cmap='terrain', origin='lower')
axes[0].set_title('White Noise (no spatial structure)', fontsize=14)
axes[0].set_xlabel('X (grid cells)')
axes[0].set_ylabel('Y (grid cells)')
plt.colorbar(im0, ax=axes[0], shrink=0.8, label='Value')

im1 = axes[1].imshow(landscape, cmap='terrain', origin='lower')
axes[1].set_title(f'Autocorrelated Landscape (σ = {SIGMA} cells)', fontsize=14)
axes[1].set_xlabel('X (grid cells)')
axes[1].set_ylabel('Y (grid cells)')
plt.colorbar(im1, ax=axes[1], shrink=0.8, label='Value')

plt.tight_layout()
plt.savefig('01_landscape_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Landscape generated and plotted.")

## 4 · Sample random points from the landscape

In a real field study you never observe the full grid — you collect *point samples*.  
Here we draw `n_samples` random locations and record the landscape value at each.

> **Sampling design note:** If your sampling grain > σ you will miss the pattern entirely.

In [ ]:
n_samples = 500
xs = np.random.randint(0, n, n_samples)
ys = np.random.randint(0, n, n_samples)
values = np.array([landscape[x, y] for x, y in zip(xs, ys)])
coords  = np.column_stack([xs, ys])

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(landscape, cmap='terrain', origin='lower', alpha=0.5)
scatter = ax.scatter(ys, xs, c=values, cmap='terrain', s=10,
                     edgecolors='black', linewidths=0.3)
ax.set_title(f'{n_samples} sample points on the landscape', fontsize=14)
ax.set_xlabel('X')
ax.set_ylabel('Y')
plt.colorbar(scatter, ax=ax, shrink=0.8, label='Sampled value')
plt.tight_layout()
plt.savefig('02_sample_locations.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ {n_samples} points sampled.")

## 5 · Calculate Moran's I at multiple lag distances

### What is Moran's I?

$$I = \frac{n}{W} \cdot \frac{\sum_i \sum_j w_{ij}(z_i - \bar{z})(z_j - \bar{z})}{\sum_i (z_i - \bar{z})^2}$$

| Symbol | Meaning |
|--------|---------|
| $n$ | number of observations |
| $w_{ij}$ | spatial weight (1 if $i$ and $j$ are neighbours, else 0) |
| $W$ | sum of all weights |
| $z_i$ | value at location $i$ |

- **I ≈ +1** → strong positive autocorrelation (similar values cluster together)  
- **I ≈ 0** → spatial independence  
- **I ≈ −1** → checkerboard pattern (dissimilar neighbours)

### Strategy: annular lag bands

Rather than cumulative disks, we use **thin annular rings** `[d−2, d]` at each lag `d`.  
This isolates the contribution of each distance, giving a cleaner correlogram.

In [ ]:
# Pre-compute all pairwise distances once (reused across all lags)
dists_all = cdist(coords, coords)

lag_centers  = np.arange(3, 50, 2)
morans_values = []
pair_counts   = []

mean_val   = np.mean(values)
deviations = values - mean_val
var_term   = np.sum(deviations**2)

print("Calculating Moran's I at multiple lag distances...")
print(f"{'Lag (cells)':>12} {'Moran\'s I':>10} {'N pairs':>10}")
print("-" * 35)

for d in lag_centers:
    # Annular band: neighbours strictly between d-2 and d
    W = (dists_all <= d) & (dists_all > d - 2) & (dists_all > 0)
    n_pairs = W.sum()

    if n_pairs < 10:
        morans_values.append(np.nan)
        pair_counts.append(n_pairs)
        continue

    numerator   = len(values) * np.sum(W * np.outer(deviations, deviations))
    denominator = n_pairs * var_term
    I = numerator / denominator

    morans_values.append(I)
    pair_counts.append(n_pairs)
    print(f"{d:>12.0f} {I:>10.4f} {n_pairs:>10}")

print("\n✓ Moran's I calculation complete.")

## 6 · Plot the spatial correlogram

The correlogram shows how spatial similarity decays with distance.  
We look for the **zero crossing** — the lag at which Moran's I drops to zero —  
which marks the boundary of spatial autocorrelation and estimates the characteristic scale.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(lag_centers, morans_values, 'o-', color='#065A82',
        markersize=6, linewidth=2, label="Moran's I (annular bands)")
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='No autocorrelation')

# Estimate zero crossing by linear interpolation
valid = [(d, m) for d, m in zip(lag_centers, morans_values) if not np.isnan(m)]
zero_crossing = None
for i in range(1, len(valid)):
    if valid[i-1][1] > 0 and valid[i][1] <= 0:
        d1, m1 = valid[i-1]
        d2, m2 = valid[i]
        zero_crossing = d1 + (0 - m1) * (d2 - d1) / (m2 - m1)
        break

if zero_crossing:
    ax.axvline(x=zero_crossing, color='#D32F2F', linestyle=':', linewidth=2,
               label=f'Estimated scale ≈ {zero_crossing:.1f} cells')

ax.axvline(x=SIGMA, color='#388E3C', linestyle='--', linewidth=2, alpha=0.7,
           label=f'True σ = {SIGMA} cells')

ax.set_xlabel('Lag distance (grid cells)', fontsize=13)
ax.set_ylabel("Moran's I", fontsize=13)
ax.set_title('Spatial Correlogram — Recovering the Characteristic Scale', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(0, 50)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('03_correlogram.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 · Summary & Interpretation

In [ ]:
print("=" * 60)
print("RESULTS")
print("=" * 60)
print(f"  True characteristic scale (σ):    {SIGMA} grid cells")
if zero_crossing:
    print(f"  Estimated scale (zero crossing):  {zero_crossing:.1f} grid cells")
    print(f"  Relative error:                   {abs(zero_crossing - SIGMA)/SIGMA * 100:.1f}%")

### Conceptual take-aways

| Observation | Meaning |
|-------------|---------|
| **Moran's I > 0 at short lags** | Nearby points have similar values → spatial clustering |
| **Moran's I → 0 at the characteristic scale** | Values become spatially independent |
| **Zero crossing ≈ 2σ** | Expected for Gaussian smoothing: autocorrelation range > kernel σ |

### Sampling design implications

- If your **sampling grain > σ**, you will miss the pattern entirely  
- If your **spatial extent < σ**, you detect trend rather than structure  
- The correlogram *shape* matters, not just the exact zero-crossing value

### In a real application

You **do not know** σ in advance. The correlogram reveals it.  
This is the core logic used in:
- Landscape ecology (patch scale detection)
- Geostatistics (variogram fitting)
- Ecology of species distributions (range of spatial dependence)

## 8 · Bonus: try different scales 🔬

Change `SIGMA` below and re-run the full notebook (Runtime → Run all) to see how the correlogram shifts.

In [ ]:
# ── Experiment here ──────────────────────────────────────────
SIGMA_experiment = 8   # try 5, 10, 20, 30 …
# ─────────────────────────────────────────────────────────────

landscape_exp = gaussian_filter(np.random.randn(n, n), sigma=SIGMA_experiment)
xs_e = np.random.randint(0, n, n_samples)
ys_e = np.random.randint(0, n, n_samples)
values_e   = np.array([landscape_exp[x, y] for x, y in zip(xs_e, ys_e)])
coords_e   = np.column_stack([xs_e, ys_e])
dists_e    = cdist(coords_e, coords_e)
dev_e      = values_e - np.mean(values_e)
var_e      = np.sum(dev_e**2)

mi_exp = []
for d in lag_centers:
    W = (dists_e <= d) & (dists_e > d - 2) & (dists_e > 0)
    np_ = W.sum()
    if np_ < 10:
        mi_exp.append(np.nan); continue
    mi_exp.append(len(values_e) * np.sum(W * np.outer(dev_e, dev_e)) / (np_ * var_e))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lag_centers, mi_exp, 'o-', color='#7B1FA2', linewidth=2,
        label=f"σ = {SIGMA_experiment} cells")
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=SIGMA_experiment, color='#F57C00', linestyle='--',
           label=f'True σ = {SIGMA_experiment}')
ax.set_xlabel('Lag distance (grid cells)', fontsize=13)
ax.set_ylabel("Moran's I", fontsize=13)
ax.set_title('Correlogram — experiment', fontsize=14)
ax.legend(); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()